# Donor Retention Propensity — EDA & Model Development

This notebook covers:
1. Data loading and cleaning
2. Exploratory data analysis (EDA)
3. Feature engineering
4. Model training, validation, and evaluation
5. Export of cleaned dataset and model artifacts

All logic here is the source of truth for `pages/1_Donor_Propensity.py` and `train_donor_model.py`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.inspection import permutation_importance

sns.set(style="whitegrid")

# Auto-detect project root
BASE_DIR = Path.cwd()
while not (BASE_DIR / "data").exists():
    if BASE_DIR == BASE_DIR.parent:
        raise FileNotFoundError("Could not find 'data' directory.")
    BASE_DIR = BASE_DIR.parent

data_dir      = BASE_DIR / "data" / "raw"
output_dir    = BASE_DIR / "outputs"
output_dir.mkdir(exist_ok=True)
charts_dir    = output_dir / "charts"
charts_dir.mkdir(exist_ok=True)

print("Project root:", BASE_DIR)


## 1. Load & Clean Data

In [ ]:
donations = pd.read_csv(data_dir / "donation_records.csv")
donors    = pd.read_csv(data_dir / "donor_profiles.csv")

# Merge
df = donations.merge(donors, on="donor_id", how="left")
df["donation_date"]   = pd.to_datetime(df["donation_date"])
df["donation_amount"] = pd.to_numeric(df["donation_amount"], errors="coerce")

print(f"Rows: {len(df):,}  |  Donors: {df['donor_id'].nunique():,}")
print(f"Date range: {df['donation_date'].min().date()} → {df['donation_date'].max().date()}")
df.head()


In [ ]:
print("Missing values:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Donation type distribution:")
print(df["donation_type"].value_counts())


## 2. Exploratory Data Analysis

In [ ]:
total_donors      = df["donor_id"].nunique()
total_revenue     = df["donation_amount"].sum()
repeat_rate       = (df.groupby("donor_id")["donation_id"].count() > 1).mean()
recurring_share   = (
    df[df["donation_type"] == "Recurring"]["donation_amount"].sum() / total_revenue
)

print(f"Total donors:           {total_donors:,}")
print(f"Total revenue:          ${total_revenue:,.2f}")
print(f"Repeat donor rate:      {repeat_rate:.1%}")
print(f"Recurring revenue share:{recurring_share:.1%}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Donation amount distribution
df["donation_amount"].clip(upper=df["donation_amount"].quantile(0.99)).hist(
    bins=40, ax=axes[0], color="#2ecc71", edgecolor="white")
axes[0].set_title("Donation Amount Distribution")
axes[0].set_xlabel("Amount ($)")

# Donations per donor
gift_counts = df.groupby("donor_id")["donation_id"].count()
gift_counts.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#3498db")
axes[1].set_title("Gifts per Donor")
axes[1].set_xlabel("Number of Gifts")

# Revenue by type
df.groupby("donation_type")["donation_amount"].sum().plot(
    kind="bar", ax=axes[2], color="#e67e22")
axes[2].set_title("Revenue by Donation Type")
axes[2].set_xlabel("")

plt.tight_layout()
plt.savefig(charts_dir / "eda_overview.png", dpi=150)
plt.show()


In [ ]:
# Referral channel and age group distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if "referral_channel" in df.columns:
    df["referral_channel"].value_counts().plot(kind="bar", ax=axes[0], color="#9b59b6")
    axes[0].set_title("Referral Channel")
    axes[0].set_xlabel("")

if "age_group" in df.columns:
    df["age_group"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#e74c3c")
    axes[1].set_title("Age Group")
    axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(charts_dir / "eda_channels.png", dpi=150)
plt.show()


In [ ]:
# Donations over time
monthly = df.resample("ME", on="donation_date")["donation_amount"].sum()
monthly.plot(figsize=(12, 4), color="#2ecc71", linewidth=2)
plt.title("Monthly Donation Revenue")
plt.ylabel("Total ($)")
plt.tight_layout()
plt.savefig(charts_dir / "monthly_revenue.png", dpi=150)
plt.show()


## 3. Feature Engineering

### Time-based split
Donations are split at the **80th percentile by date**:
- **Training window** (80%) — used to build all features
- **Future window** (20%) — used only to define the target (`donated_again`)

This mirrors real deployment: the model is trained on historical data and scores current donors.

### Target
`donated_again = 1` if the donor made at least one donation after the cutoff date.

### Features
All features are derived from the **training window only** to prevent leakage.


In [ ]:
cutoff_date = df["donation_date"].quantile(0.8)
train_df    = df[df["donation_date"] <= cutoff_date]
future_df   = df[df["donation_date"] >  cutoff_date]

print(f"Training window: {train_df['donation_date'].min().date()} → {cutoff_date.date()}")
print(f"Future window:   {cutoff_date.date()} → {future_df['donation_date'].max().date()}")
print(f"Training rows:   {len(train_df):,}")
print(f"Future rows:     {len(future_df):,}")


In [ ]:
def build_donor_features(transactions: pd.DataFrame, ref_date: pd.Timestamp) -> pd.DataFrame:
    """
    Build per-donor features from transaction history.
    Mirrors build_donor_features() in train_donor_model.py exactly.
    """
    g = transactions.groupby("donor_id")

    summary = g.agg(
        donation_count =("donation_id",    "count"),
        total_donated  =("donation_amount", "sum"),
        avg_donation   =("donation_amount", "mean"),
        last_donation  =("donation_date",   "max"),
        first_donation =("donation_date",   "min"),
    ).reset_index()

    summary["recency_days"]            = (ref_date - summary["last_donation"]).dt.days
    summary["months_since_last"]       = summary["recency_days"] / 30.44
    summary["days_since_first"]        = (ref_date - summary["first_donation"]).dt.days
    summary["amount_log"]              = np.log1p(summary["total_donated"])
    summary["avg_donation_log"]        = np.log1p(summary["avg_donation"])
    active_days                         = summary["days_since_first"].replace(0, 1)
    summary["giving_frequency_ratio"]  = summary["donation_count"] / active_days
    summary["first_gift_missing_flag"] = summary["first_donation"].isna().astype(int)
    summary["never_donated_flag"]      = 0

    lifetime = transactions.groupby("donor_id").agg(
        gift_100_lifetime=("donation_amount", lambda x: int((x >= 100).any())),
    ).reset_index()
    summary = summary.merge(lifetime, on="donor_id", how="left")

    if "newsletter_opt_in" in transactions.columns:
        nl = (transactions.sort_values("donation_date")
              .groupby("donor_id")["newsletter_opt_in"].last().reset_index())
        nl["newsletter_opt_in"] = nl["newsletter_opt_in"].astype(int)
        summary = summary.merge(nl, on="donor_id", how="left")
        summary["newsletter_opt_in"] = summary["newsletter_opt_in"].fillna(0).astype(int)

    if "referral_channel" in transactions.columns:
        ref = transactions.groupby("donor_id")["referral_channel"].agg(
            lambda x: x.value_counts().index[0]
        ).reset_index()
        dummies = pd.get_dummies(ref["referral_channel"], prefix="ref").astype(int)
        ref = pd.concat([ref[["donor_id"]], dummies], axis=1)
        summary = summary.merge(ref, on="donor_id", how="left")
        for c in [col for col in summary.columns if col.startswith("ref_")]:
            summary[c] = summary[c].fillna(0).astype(int)

    if "age_group" in transactions.columns:
        age = transactions.groupby("donor_id")["age_group"].agg(
            lambda x: x.value_counts().index[0]
        ).reset_index()
        dummies = pd.get_dummies(age["age_group"], prefix="age").astype(int)
        age = pd.concat([age[["donor_id"]], dummies], axis=1)
        summary = summary.merge(age, on="donor_id", how="left")
        for c in [col for col in summary.columns if col.startswith("age_")]:
            summary[c] = summary[c].fillna(0).astype(int)

    return summary


donor_summary = build_donor_features(train_df, cutoff_date)
future_donors = future_df["donor_id"].unique()
donor_summary["donated_again"] = donor_summary["donor_id"].isin(future_donors).astype(int)

print(f"Donors: {len(donor_summary):,}")
print(f"Retention rate: {donor_summary['donated_again'].mean():.1%}")
donor_summary.head()


In [ ]:
BASE_FEATURES = [
    "recency_days", "donation_count", "amount_log", "avg_donation_log",
    "months_since_last", "days_since_first", "giving_frequency_ratio",
    "gift_100_lifetime", "first_gift_missing_flag", "never_donated_flag",
    "newsletter_opt_in",
]
EXTRA_FEATURES = [c for c in donor_summary.columns
                  if c.startswith("ref_") or c.startswith("age_")]
MODEL_FEATURES = [f for f in BASE_FEATURES + EXTRA_FEATURES if f in donor_summary.columns]

X = donor_summary[MODEL_FEATURES].fillna(0)
y = donor_summary["donated_again"]

print(f"Features used: {len(MODEL_FEATURES)}")
print(MODEL_FEATURES)


## 4. Model Training & Validation

### Model
`RandomForestClassifier` with `class_weight="balanced_subsample"` to handle the class
imbalance (~12% retention rate). Time-based train/test split preserves temporal order.

### Validation
- **5-fold stratified cross-validation** on the full dataset for a robust ROC-AUC estimate
- **Held-out time-split test set** for PR-AUC and Recall@K — the most operationally relevant metrics


In [ ]:
# Time-based train/test split within donor_summary
split_cutoff    = donor_summary["last_donation"].quantile(0.8)
train_mask      = donor_summary["last_donation"] <= split_cutoff
test_mask       = donor_summary["last_donation"] >  split_cutoff
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train):,} donors | Positive rate: {y_train.mean():.1%}")
print(f"Test:  {len(X_test):,} donors  | Positive rate: {y_test.mean():.1%}")


In [ ]:
RANDOM_STATE = 42

rf = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# 5-fold stratified cross-validation — robust ROC-AUC estimate
cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_aucs = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
roc_auc_cv = cv_aucs.mean()

print(f"CV ROC-AUC: {roc_auc_cv:.4f} ± {cv_aucs.std():.4f}")
print(f"Per-fold:   {[f'{v:.3f}' for v in cv_aucs]}")


In [ ]:
# Final model fit on time-split training window
rf.fit(X_train, y_train)
p_test = rf.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, p_test)

def recall_at_k(y_true, y_score, k):
    idx = np.argsort(-y_score)[:max(1, int(len(y_true) * k))]
    return y_true.iloc[idx].sum() / max(1, y_true.sum())

print(f"PR-AUC:         {pr_auc:.4f}")
print(f"Recall @ top 10%: {recall_at_k(y_test, p_test, 0.10):.1%}")
print(f"Recall @ top 20%: {recall_at_k(y_test, p_test, 0.20):.1%}")
print(f"Recall @ top 33%: {recall_at_k(y_test, p_test, 0.33):.1%}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[0], color="#2ecc71")
axes[0].plot([0,1],[0,1],"--", color="grey", linewidth=1)
axes[0].set_title(f"ROC Curve (AUC = {roc_auc_cv:.3f} CV)")

PrecisionRecallDisplay.from_predictions(y_test, p_test, ax=axes[1], color="#3498db")
axes[1].set_title(f"Precision-Recall Curve (AP = {pr_auc:.3f})")

plt.tight_layout()
plt.savefig(charts_dir / "model_evaluation.png", dpi=150)
plt.show()


## 5. Lift Chart

In [ ]:
lift_df = pd.DataFrame({"score": p_test, "target": y_test.values})
lift_df = lift_df.sort_values("score", ascending=False).reset_index(drop=True)
total   = len(lift_df)
total_r = lift_df["target"].sum()

steps         = list(range(1, 101))
model_capture = [lift_df["target"].iloc[:max(1, int(total*p/100))].sum() / total_r * 100 for p in steps]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps, model_capture, color="#2ecc71", linewidth=2, label="Model")
ax.plot(steps, steps,         color="#aaaaaa", linewidth=1.5, linestyle="--", label="Random")
ax.set_xlabel("% of Donors Contacted")
ax.set_ylabel("% of Retained Donors Captured")
ax.set_title("Model vs Random — Lift Chart (test set)")
ax.legend()

top33 = lift_df["target"].iloc[:int(total*0.33)].sum() / total_r * 100
ax.annotate(f"Top 33%: {top33:.1f}% captured\n({top33/33:.1f}x lift)",
            xy=(33, top33), xytext=(45, top33-15),
            arrowprops=dict(arrowstyle="->", color="#333"), fontsize=10)

plt.tight_layout()
plt.savefig(charts_dir / "lift_chart.png", dpi=150)
plt.show()
print(f"Lift at top 33%: {top33/33:.2f}x  |  Retained captured: {top33:.1f}%")


## 6. Feature Importance

In [ ]:
rng   = np.random.RandomState(RANDOM_STATE)
idx   = rng.choice(len(X_test), min(300, len(X_test)), replace=False)
perm  = permutation_importance(
    rf, X_test.iloc[idx], y_test.iloc[idx],
    n_repeats=5, random_state=RANDOM_STATE, scoring="average_precision",
)

imp_df = pd.DataFrame({
    "feature":    X_test.columns,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df["feature"][::-1], imp_df["importance"][::-1], color="#3498db", alpha=0.85)
ax.set_xlabel("Mean Importance (PR-AUC drop)")
ax.set_title("Permutation Feature Importance")
plt.tight_layout()
plt.savefig(charts_dir / "feature_importance.png", dpi=150)
plt.show()

print("\nTop 10 features:")
print(imp_df.head(10).to_string(index=False))


## 7. Segmentation & Retention by Tier

In [ ]:
# Score all donors and apply 4-tier segmentation
donor_summary["propensity_score"] = rf.predict_proba(X)[:, 1]

q80, q60, q40 = (donor_summary["propensity_score"].quantile(q)
                 for q in [0.80, 0.60, 0.40])

def assign_tier(s):
    if s >= q80:   return "High"
    elif s >= q60: return "Medium"
    elif s >= q40: return "Low"
    else:          return "Very Low"

donor_summary["segment"] = donor_summary["propensity_score"].apply(assign_tier)

print("Segment distribution:")
print(donor_summary["segment"].value_counts())
print()
print("Retention rate by segment:")
print(donor_summary.groupby("segment")["donated_again"].mean().sort_values(ascending=False).round(3))


In [ ]:
seg_order  = ["High", "Medium", "Low", "Very Low"]
seg_colors = {"High": "#2ecc71", "Medium": "#f39c12", "Low": "#e67e22", "Very Low": "#e74c3c"}

retention = (donor_summary.groupby("segment")["donated_again"]
             .mean().reindex(seg_order) * 100)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(retention.index, retention.values,
              color=[seg_colors[s] for s in retention.index], alpha=0.85)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_ylabel("Retention Rate (%)")
ax.set_title("Retention Rate by Propensity Tier")
ax.set_ylim(0, retention.max() * 1.2)
plt.tight_layout()
plt.savefig(charts_dir / "retention_by_tier.png", dpi=150)
plt.show()


## 8. Export

Save the cleaned dataset. The model artifact is generated by `train_donor_model.py`
which follows the same feature engineering logic defined in this notebook.
Run `uv run python train_donor_model.py` after updating this notebook to refresh
the deployed model.


In [ ]:
df.to_csv(output_dir / "donations_clean.csv", index=False)
print(f"Saved: {output_dir / 'donations_clean.csv'}")
print(f"       {len(df):,} rows  |  {df['donor_id'].nunique():,} donors")
print()
print("Next step: run  uv run python train_donor_model.py  to rebuild the model artifact.")
